In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_store = spark.table("workspace.bronze.store")
df_store = normalize_columns(df_store)

df_store = (
    df_store
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("store_id", F.col("store_id").cast("int"))
    .withColumn("opened_year", F.col("opened_year").cast("int"))
    .dropDuplicates()
)


manager_lookup = (
    df_store.filter(F.col("manager_name").isNotNull())
    .select("manager_id", F.col("manager_name").alias("manager_name_lookup")).distinct()
)
df_store = (
    df_store
    .join(manager_lookup, on="manager_id", how="left")
    .withColumn("manager_name", F.coalesce(F.col("manager_name"), F.col("manager_name_lookup")))
    .drop("manager_name_lookup")
)

df_store.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.store")
print(f"silver.store: {df_store.count()} rows written")